# Cleaning the session logs

Removes session directories from the logs whose **student ID does not appear in the questionnaire** `results.csv` — they are test / dev / aborted runs that should not be considered experimental data.


## Section 0 — Paths and config

In [31]:
import csv
import re
import shutil
import collections
import pathlib
from datetime import datetime

# Resolve from the project root so the notebook works from anywhere.
PROJECT_ROOT = pathlib.Path.cwd()
while not (PROJECT_ROOT / "docker").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

LOG_ROOT_SRC = PROJECT_ROOT / "voice-agent/src/experiment/results/experiments"
LOG_ROOT_DST = PROJECT_ROOT / "voice-agent/src/experiment/results/experiments_cleaned"
CSV_PATH     = PROJECT_ROOT / "docs/thesis/experiment/results/results.csv"
MANIFEST_PATH = PROJECT_ROOT / "docs/thesis/experiment/results/cleaning_manifest.csv"

# Set to True only after reviewing the to-delete list in Section 3.
CONFIRM_DELETION = True
# Section 7 — set to True to also delete sessions that had no user_turn / agent_speech events.
CONFIRM_DELETE_EMPTY = True
# Also remove date folders that end up empty after deletion.
REMOVE_EMPTY_DATE_DIRS = True

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"SRC          : {LOG_ROOT_SRC}  (exists={LOG_ROOT_SRC.exists()})")
print(f"DST          : {LOG_ROOT_DST}  (exists={LOG_ROOT_DST.exists()})")
print(f"CSV          : {CSV_PATH}      (exists={CSV_PATH.exists()})")
print(f"MANIFEST     : {MANIFEST_PATH}")
print()
print(f"CONFIRM_DELETION       = {CONFIRM_DELETION}")
print(f"CONFIRM_DELETE_EMPTY   = {CONFIRM_DELETE_EMPTY}")
print(f"REMOVE_EMPTY_DATE_DIRS = {REMOVE_EMPTY_DATE_DIRS}")

PROJECT_ROOT : /home/lucas/Projects/FEL/Pepper
SRC          : /home/lucas/Projects/FEL/Pepper/voice-agent/src/experiment/results/experiments  (exists=True)
DST          : /home/lucas/Projects/FEL/Pepper/voice-agent/src/experiment/results/experiments_cleaned  (exists=True)
CSV          : /home/lucas/Projects/FEL/Pepper/docs/thesis/experiment/results/results.csv      (exists=True)
MANIFEST     : /home/lucas/Projects/FEL/Pepper/docs/thesis/experiment/results/cleaning_manifest.csv

CONFIRM_DELETION       = True
CONFIRM_DELETE_EMPTY   = True
REMOVE_EMPTY_DATE_DIRS = True


## Section 1 — Make a copy of the logs

If `LOG_ROOT_DST` already exists the copy is skipped. To get a fresh copy from scratch, delete that directory first.

In [32]:
if LOG_ROOT_DST.exists():
    print(f"DST already exists: {LOG_ROOT_DST}")
    print(f"  Skipping copy. Delete this directory first if you want a fresh copy.")
else:
    print(f"Copying {LOG_ROOT_SRC} -> {LOG_ROOT_DST} ...")
    shutil.copytree(LOG_ROOT_SRC, LOG_ROOT_DST)
    print(f"Done.")

n_sessions = sum(1 for d in LOG_ROOT_DST.glob("*/student*_streaming*_*") if d.is_dir())
n_dates = len([d for d in LOG_ROOT_DST.iterdir() if d.is_dir()])
print(f"\nCopy contents: {n_sessions} session dirs across {n_dates} date folders.")

DST already exists: /home/lucas/Projects/FEL/Pepper/voice-agent/src/experiment/results/experiments_cleaned
  Skipping copy. Delete this directory first if you want a fresh copy.

Copy contents: 38 session dirs across 1 date folders.


## Section 2 — Collect the questionnaire IDs

Each row in `results.csv` has a `Conversation ID:` like `T63`, `d45`, `K01`. We extract the numeric portion (`T63` → `63`, `d45` → `45`) and build the set of IDs that correspond to real respondents. Sessions whose student ID is in this set are kept; everything else is deletable.

In [33]:
keep_ids = set()
with open(CSV_PATH) as f:
    for row in csv.DictReader(f):
        conv = row["Conversation ID:"]
        m = re.search(r"\d+", conv)
        if m:
            keep_ids.add(int(m.group()))

print(f"Questionnaire has {len(keep_ids)} unique student IDs:")
print(f"  {sorted(keep_ids)}")

Questionnaire has 18 unique student IDs:
  [1, 7, 8, 14, 30, 33, 34, 38, 40, 44, 45, 47, 49, 50, 52, 58, 61, 63]


## Section 3 — Identify sessions to delete

A session is kept if its student ID matches one in the questionnaire. Anything else is a test / dev / aborted-by-experimenter run and is marked for deletion. Directory names that do not match the expected `student<id>_streaming<A|B>_<HHMMSS>` pattern are conservatively **kept** (to avoid accidentally deleting something unrecognised).

In [34]:
to_delete = []
to_keep = []
unrecognised = []

for d in sorted(LOG_ROOT_DST.glob("*/student*_streaming*_*")):
    if not d.is_dir():
        continue
    m = re.match(r"student(\d+)_streaming([AB])_(\d{6})$", d.name)
    if not m:
        unrecognised.append(d)
        continue
    sid = int(m.group(1))
    if sid in keep_ids:
        to_keep.append((d, sid))
    else:
        to_delete.append((d, sid))

print(f"Sessions to KEEP   : {len(to_keep)}")
print(f"Sessions to DELETE : {len(to_delete)}")
print(f"Unrecognised names : {len(unrecognised)}  (kept defensively)")

if to_delete:
    deleted_ids = collections.Counter(sid for _, sid in to_delete)
    print(f"\n=== Per-ID counts for DELETE (top 20 most-used test IDs) ===")
    for sid, n in deleted_ids.most_common(20):
        print(f"  student{sid:<4}  {n} session dir(s)")
    print(f"\n=== Sample of TO DELETE (first 30) ===")
    for d, sid in to_delete[:30]:
        print(f"  student{sid:<4}  {d.relative_to(LOG_ROOT_DST)}")
    if len(to_delete) > 30:
        print(f"  ... and {len(to_delete) - 30} more")

if unrecognised:
    print(f"\n=== Unrecognised (kept) ===")
    for d in unrecognised:
        print(f"  {d.relative_to(LOG_ROOT_DST)}")

Sessions to KEEP   : 38
Sessions to DELETE : 0
Unrecognised names : 0  (kept defensively)


## Section 4 — Delete (gated by `CONFIRM_DELETION`)

Runs only if `CONFIRM_DELETION = True` in Section 0. Otherwise this cell prints what would happen and exits. Every deletion is appended to `cleaning_manifest.csv`.

In [35]:
if not CONFIRM_DELETION:
    print(f"CONFIRM_DELETION = False. No files were touched.")
    print(f"To execute the deletion, set CONFIRM_DELETION = True in Section 0 and re-run from this cell.")
else:
    timestamp = datetime.now().isoformat(timespec="seconds")
    write_header = not MANIFEST_PATH.exists()
    with open(MANIFEST_PATH, "a", newline="") as f:
        w = csv.writer(f)
        if write_header:
            w.writerow(["timestamp", "dir", "student_id", "reason"])
        for d, sid in to_delete:
            w.writerow([timestamp,
                       str(d.relative_to(LOG_ROOT_DST)),
                       sid,
                       "student_id not in questionnaire"])

    n = 0
    for d, sid in to_delete:
        shutil.rmtree(d)
        n += 1

    n_emptied = 0
    if REMOVE_EMPTY_DATE_DIRS:
        for date_dir in sorted(LOG_ROOT_DST.iterdir()):
            if date_dir.is_dir() and not any(date_dir.iterdir()):
                date_dir.rmdir()
                n_emptied += 1

    print(f"Deleted {n} session directories from {LOG_ROOT_DST}.")
    print(f"Removed {n_emptied} empty date folders.")
    print(f"Manifest appended: {MANIFEST_PATH}")

Deleted 0 session directories from /home/lucas/Projects/FEL/Pepper/voice-agent/src/experiment/results/experiments_cleaned.
Removed 0 empty date folders.
Manifest appended: /home/lucas/Projects/FEL/Pepper/docs/thesis/experiment/results/cleaning_manifest.csv


## Section 5 — Post-cleaning summary

What remains in the cleaned directory, broken down by date, variant, and student ID.

In [36]:
remaining = sorted(d for d in LOG_ROOT_DST.glob("*/student*_streaming*_*") if d.is_dir())
print(f"Sessions remaining in {LOG_ROOT_DST}: {len(remaining)}")

per_date = collections.Counter()
per_variant = collections.Counter()
per_id = collections.Counter()
for d in remaining:
    m = re.match(r"student(\d+)_streaming([AB])_\d{6}$", d.name)
    if m:
        per_id[int(m.group(1))] += 1
        per_variant[m.group(2)] += 1
    per_date[d.parent.name] += 1

print("\nBy date:")
for k, v in sorted(per_date.items()):
    print(f"  {k}: {v}")

print("\nBy variant:")
for k, v in sorted(per_variant.items()):
    print(f"  Condition {k}: {v}")

print("\nBy student id:")
for sid in sorted(per_id):
    print(f"  student{sid:<4}: {per_id[sid]}")

Sessions remaining in /home/lucas/Projects/FEL/Pepper/voice-agent/src/experiment/results/experiments_cleaned: 38

By date:
  2026-05-18: 38

By variant:
  Condition A: 22
  Condition B: 16

By student id:
  student1   : 1
  student7   : 1
  student8   : 1
  student14  : 1
  student30  : 1
  student33  : 1
  student34  : 3
  student38  : 2
  student40  : 1
  student44  : 2
  student45  : 3
  student47  : 1
  student49  : 4
  student50  : 3
  student52  : 1
  student58  : 2
  student61  : 4
  student63  : 6


## Section 6 — Reconstruct conversations from the remaining sessions

For each session that survived cleaning, walks its `events.jsonl` and writes a `transcript.txt` **inside the same session directory** (next to `events.jsonl` and `audio/`). The transcript also prints inline in the notebook so you can scan it without leaving Jupyter.

What ends up in the transcript:

- `[user (speech)]` — what the visitor said (transcribed by STT).
- `[user (typed)]` — what the experimenter typed into the operator data channel during the session.
- `[pepper]` — what Pepper actually spoke (the `text` argument of `send_message_to_user` / `end_conversation`).

Everything else (VAD, LLM internals, gestures, tablet state) is skipped — this is just the bare dialogue, the easiest thing to scan to spot weird turns, false answers, or moments where the visitor lost the thread.

In [37]:
import json
sessions = sorted(d for d in LOG_ROOT_DST.glob("*/student*_streaming*_*") if d.is_dir())
print(f"Reconstructing {len(sessions)} sessions; writing transcript.txt into each session directory...")
print()

n_written = 0
for d in sessions:
    jl = d / "events.jsonl"
    if not jl.exists():
        print(f"!! no events.jsonl in {d.relative_to(LOG_ROOT_DST)} -- skipping")
        continue

    # collect header + dialogue events
    header = None
    dialogue = []  # list of (ts, kind, text)  kind in {"user_speech", "user_typed", "pepper"}
    footer = None
    for ln in jl.read_text(errors="replace").splitlines():
        if not ln.strip(): continue
        try: ev = json.loads(ln)
        except json.JSONDecodeError: continue
        et = ev.get("event")
        data = ev.get("data") or {}
        if et == "header":
            header = data
        elif et == "footer":
            footer = data
        elif et == "user_turn":
            text = (data.get("text") or "").strip()
            if text:
                kind = "user_typed" if data.get("input") == "typed" else "user_speech"
                dialogue.append((ev.get("ts", 0), kind, text))
        elif et == "agent_speech":
            text = (data.get("text") or "").strip()
            if text:
                dialogue.append((ev.get("ts", 0), "pepper", text))

    dialogue.sort(key=lambda x: x[0])

    # header info
    conv_id = (footer or {}).get("conv_id") or (header or {}).get("student_id") or "?"
    variant = (footer or {}).get("variant") or (header or {}).get("variant", "?").replace("streaming", "")
    started = (header or {}).get("started") or "?"
    n_turns = (footer or {}).get("n_user_turns", "?")
    dur = (footer or {}).get("duration_seconds")
    dur_str = f"{dur:.1f}s" if isinstance(dur, (int, float)) else "?"

    # build the transcript text
    lines = []
    lines.append("=" * 78)
    lines.append(d.name)
    lines.append(f"  conv_id={conv_id}  variant={variant}  started={started}  user_turns={n_turns}  duration={dur_str}")
    lines.append("=" * 78)
    if not dialogue:
        lines.append("  (no user_turn or agent_speech events in this session)")
    for _ts, kind, text in dialogue:
        label = {"user_speech": "user (speech)", "user_typed": "user (typed)", "pepper": "pepper"}[kind]
        lines.append(f"  [{label}]  {text}")

    transcript_text = "\n".join(lines)

    # write to the session directory
    out_path = d / "transcript.txt"
    out_path.write_text(transcript_text + "\n")
    n_written += 1

    # echo to notebook output too
    print(transcript_text)
    print(f"  -> wrote {out_path.relative_to(LOG_ROOT_DST)}")
    print()

print(f"Done. Wrote transcript.txt into {n_written} session directories under {LOG_ROOT_DST}.")

Reconstructing 38 sessions; writing transcript.txt into each session directory...

student14_streamingB_100523
  conv_id=T14  variant=B  started=2026-05-18T10:05:23  user_turns=4  duration=295.5s
  [user (speech)]  Ahoj, jak se dostanu do místnosti tři sta jedna?
  [pepper]  Ahoj! Místnost 301 neznám, ale mohu ti pomoci s informacemi o jiných místnostech nebo zařízeních. Jak ti mohu pomoci?
  [user (speech)]  Potřebuji se dostat do místnosti přednáškové.
  [pepper]  Let me find that room for you.
  [pepper]  Jdi nahoru na první patro a pokračuj rovně — místnost E-107 je podél toho chodníku.
  [user (speech)]  Dobře, díky.
  [pepper]  Není zač! Kdybys potřeboval další pomoc, dej vědět.
  [user (speech)]  Už nic dalšího nepotřebuje.
  [pepper]  Měj se hezky!
  [pepper]  Your conversation ID is T one four.
  [pepper]  Please help me improve by scanning the QR code on my tablet or on the board and filling the questionnaire!
  -> wrote 2026-05-18/student14_streamingB_100523/transcript.txt



## Section 7 — Delete sessions with no real dialogue

After running Section 6 you may discover sessions whose `transcript.txt` reads:

> (no user_turn or agent_speech events in this session)

These are dispatched-and-abandoned sessions — the worker spun up, the visitor never spoke, the session ended. They carry no useful data and should be removed.

The deletion runs only if `CONFIRM_DELETE_EMPTY = True` in Section 0. Otherwise this cell lists the empty sessions but does not touch them. Deletions are appended to the same `cleaning_manifest.csv` (with a different `reason`).

In [38]:
empty_sessions = []
remaining = sorted(d for d in LOG_ROOT_DST.glob("*/student*_streaming*_*") if d.is_dir())

for d in remaining:
    jl = d / "events.jsonl"
    if not jl.exists():
        empty_sessions.append((d, "no events.jsonl"))
        continue
    has_dialogue = False
    for ln in jl.read_text(errors="replace").splitlines():
        if not ln.strip(): continue
        try: ev = json.loads(ln)
        except json.JSONDecodeError: continue
        et = ev.get("event")
        data = ev.get("data") or {}
        if et in ("user_turn", "agent_speech"):
            if (data.get("text") or "").strip():
                has_dialogue = True
                break
    if not has_dialogue:
        empty_sessions.append((d, "no user_turn or agent_speech with text"))

print(f"Sessions with no dialogue: {len(empty_sessions)}")
if empty_sessions:
    print()
    print("=== Sessions to DELETE (empty) ===")
    for d, reason in empty_sessions:
        print(f"  {d.relative_to(LOG_ROOT_DST)}   ({reason})")

if not CONFIRM_DELETE_EMPTY:
    print()
    print(f"CONFIRM_DELETE_EMPTY = False. Nothing deleted.")
    print(f"Set CONFIRM_DELETE_EMPTY = True in Section 0 and re-run this cell to execute the deletion.")
else:
    timestamp = datetime.now().isoformat(timespec="seconds")
    write_header = not MANIFEST_PATH.exists()
    with open(MANIFEST_PATH, "a", newline="") as f:
        w = csv.writer(f)
        if write_header:
            w.writerow(["timestamp", "dir", "student_id", "reason"])
        for d, reason in empty_sessions:
            m = re.match(r"student(\d+)_", d.name)
            sid = int(m.group(1)) if m else -1
            w.writerow([timestamp, str(d.relative_to(LOG_ROOT_DST)), sid, reason])

    for d, _ in empty_sessions:
        shutil.rmtree(d)

    n_emptied = 0
    if REMOVE_EMPTY_DATE_DIRS:
        for date_dir in sorted(LOG_ROOT_DST.iterdir()):
            if date_dir.is_dir() and not any(date_dir.iterdir()):
                date_dir.rmdir()
                n_emptied += 1

    print()
    print(f"Deleted {len(empty_sessions)} empty session directories.")
    print(f"Removed {n_emptied} now-empty date folders.")
    print(f"Manifest appended: {MANIFEST_PATH}")

Sessions with no dialogue: 3

=== Sessions to DELETE (empty) ===
  2026-05-18/student50_streamingA_153646   (no user_turn or agent_speech with text)
  2026-05-18/student63_streamingA_172849   (no user_turn or agent_speech with text)
  2026-05-18/student63_streamingB_172410   (no user_turn or agent_speech with text)

Deleted 3 empty session directories.
Removed 0 now-empty date folders.
Manifest appended: /home/lucas/Projects/FEL/Pepper/docs/thesis/experiment/results/cleaning_manifest.csv


## Section 8 — Manual review and deletion of failed interactions

After the automated cleaning in Sections 4 and 7, the remaining sessions were reviewed by hand. For each surviving session the `transcript.txt` produced in Section 6 was read end to end, and any session that clearly failed as a real interaction — for example a visitor walking away after one confused exchange, the agent stuck in a tool-call loop, or speech recognition mis-transcribing every utterance into nonsense — was removed from the working set.



The output of this manual pass is kept separately at:

```
voice-agent/src/experiment/results/experiments_cleaned_manualy/
```

so that `experiments_cleaned/` remains as the post-automatic-cleaning checkpoint and the manual deletions can be audited at any time by comparing the two directories (for example with `diff -r`).

Downstream analysis (`analysis.ipynb`, the matching notebook, etc.) should point at `experiments_cleaned_manualy/` as the final, study-grade session set.